# COLETA VOTAÇÕES


In [ ]:
import pandas as pd

def gerar_arquivao_presenca(anos):
    frames_presenca = []

    for ano in anos:
        print(f"📥 Baixando dados de {ano}...")
        url_votos = f"https://dadosabertos.camara.leg.br/arquivos/votacoesVotos/csv/votacoesVotos-{ano}.csv"
        url_pautas = f"https://dadosabertos.camara.leg.br/arquivos/votacoes/csv/votacoes-{ano}.csv"

        try:
            df_votos = pd.read_csv(url_votos, sep=';', low_memory=False)
            presencas = df_votos.groupby(['deputado_id', 'deputado_nome']).size().reset_index(name='presencas_nominais')
            nominais_ano = df_votos['idVotacao'].nunique()

            df_pautas = pd.read_csv(url_pautas, sep=';', low_memory=False)
            total_reunioes = len(df_pautas)

            presencas['total_nominais_ano'] = nominais_ano
            presencas['total_geral_reunioes_ano'] = total_reunioes

            frames_presenca.append(presencas)

        except Exception as e:
            print(f" Erro no ano {ano}: {e}")

    print("\n Unindo todos os anos e calculando a carreira...")
    df_completo = pd.concat(frames_presenca, ignore_index=True)


    ranking = df_completo.groupby('deputado_id').agg({
        'deputado_nome': 'first',
        'presencas_nominais': 'sum',
        'total_nominais_ano': 'sum',
        'total_geral_reunioes_ano': 'sum'
    }).reset_index()

    ranking['taxa_assiduidade_%'] = (ranking['presencas_nominais'] / ranking['total_nominais_ano']) * 100
    ranking['taxa_assiduidade_%'] = ranking['taxa_assiduidade_%'].round(2)

    return ranking

anos_historicos = list(range(2000, 2027))
df_modulo2 = gerar_arquivao_presenca(anos_historicos)
df_modulo2.to_csv("modulo2_presenca_completa.csv", index=False)
print("\n Módulo 2 sucesso! Arquivo gerado.")